# EMG &rarr; Hand-Kinematics Prediction with Self-Supervised Learning

Predicts continuous hand kinematics (CyberGlove joint values) from forearm EMG,
using masked-reconstruction SSL pretraining, then evaluates it.

This notebook is the **algorithm + evaluation** — the graded core of the brief:
* **Obj. 3** — predict hand kinematics from muscle activity (this is *regression* on joint angles, not gesture classification).
* **Obj. 4** — evaluate performance, cross-subject, with a noise-robustness sweep.

**Run order:** top to bottom. The two key experiments are *from-scratch* vs *SSL-pretrained*, compared on held-out subjects.

### Before you start
1. Register at https://ninapro.hevs.ch and download **DB2**.
2. Put the per-subject `.mat` files (e.g. `S1_E1_A1.mat`, `S2_E1_A1.mat`, ...) in a `data/` folder next to this notebook.
3. Real training needs a GPU and takes a while — first set the epochs low in the Config cell to smoke-test the whole flow, then scale up.

## 1. Setup

In [1]:
# Uncomment to install dependencies if needed:
# %pip install torch numpy scipy matplotlib

import os, glob, re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.io import loadmat
import matplotlib.pyplot as plt

## 2. Configuration
Edit everything here — paths, window size, epochs, and which subjects are held out for testing.

In [2]:
from dataclasses import dataclass

@dataclass
class Config:
    # data
    data_dir: str = "data"      # folder of Ninapro DB2 .mat files
    fs: int = 2000              # DB2 sampling rate (Hz)
    n_emg: int = 12             # DB2 EMG channels
    n_kin: int = 22             # CyberGlove kinematic channels (targets)
    win_ms: int = 200           # analysis window length
    stride_ms: int = 50         # hop between windows
    # model
    feat_dim: int = 128
    # SSL pretraining (masked reconstruction)
    mask_ratio: float = 0.4
    ssl_epochs: int = 40
    ssl_lr: float = 1e-3
    # supervised regression
    reg_epochs: int = 40
    reg_lr: float = 1e-3
    batch_size: int = 128
    # cross-subject split: these subjects are held out for testing only
    test_subjects: tuple = (1, 2)
    seed: int = 0
    device: str = "cuda"        # falls back to cpu automatically

cfg = Config()
print(cfg)

Config(data_dir='data', fs=2000, n_emg=12, n_kin=22, win_ms=200, stride_ms=50, feat_dim=128, mask_ratio=0.4, ssl_epochs=40, ssl_lr=0.001, reg_epochs=40, reg_lr=0.001, batch_size=128, test_subjects=(1, 2), seed=0, device='cuda')


## 3. Utilities
Reproducibility, the kinematics metrics used throughout, and SNR noise injection.

In [3]:
def set_seed(seed=0):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def get_device(pref="cuda"):
    return torch.device(pref if (pref == "cuda" and torch.cuda.is_available()) else "cpu")

# ---- regression metrics (on joint angles) ----
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2, axis=0)
    ss_tot = np.sum((y_true - y_true.mean(0)) ** 2, axis=0) + 1e-12
    return float(np.mean(1 - ss_res / ss_tot))

def mean_corr(y_true, y_pred):
    """Average Pearson correlation across kinematic channels."""
    corrs = []
    for j in range(y_true.shape[1]):
        a, b = y_true[:, j], y_pred[:, j]
        if a.std() < 1e-8 or b.std() < 1e-8:
            continue
        corrs.append(np.corrcoef(a, b)[0, 1])
    return float(np.mean(corrs)) if corrs else 0.0

def add_noise_snr(emg, snr_db):
    """Add Gaussian noise at a target SNR (dB) to an EMG array (n, ch, T)."""
    sig_power = np.mean(emg ** 2)
    noise_power = sig_power / (10 ** (snr_db / 10))
    noise = np.random.randn(*emg.shape) * np.sqrt(noise_power)
    return (emg + noise).astype(np.float32)

set_seed(cfg.seed)
device = get_device(cfg.device)
print("device:", device)

device: cpu


## 4. Data

Loads Ninapro DB2 and windows it. The task is **regression**: given a window of EMG, predict the hand kinematics (glove vector) at the end of that window.

DB2 `.mat` fields used: `emg` (12ch, input), `glove` (22ch, the kinematics target), `subject` (for cross-subject splitting).

In [4]:
def _subject_id(mat, path):
    if "subject" in mat:
        return int(np.asarray(mat["subject"]).ravel()[0])
    m = re.search(r"[Ss](\d+)", os.path.basename(path))
    return int(m.group(1)) if m else 0

def load_files(data_dir):
    """Read every .mat file. Returns list of (emg, glove, subject)."""
    paths = sorted(glob.glob(os.path.join(data_dir, "*.mat")))
    if not paths:
        raise FileNotFoundError(f"No .mat files in '{data_dir}'. Download Ninapro DB2 first.")
    recordings = []
    for p in paths:
        m = loadmat(p)
        emg = m["emg"].astype(np.float32)
        glove = m["glove"].astype(np.float32)
        recordings.append((emg, glove, _subject_id(m, p)))
    print(f"loaded {len(recordings)} recordings, subjects: {sorted({s for *_, s in recordings})}")
    return recordings

def window_recording(emg, glove, fs, win_ms, stride_ms):
    """Slide windows; target = kinematics at the window's last sample."""
    win = int(fs * win_ms / 1000)
    hop = int(fs * stride_ms / 1000)
    X, Y = [], []
    for s in range(0, len(emg) - win, hop):
        X.append(emg[s:s + win].T)        # (channels, time)
        Y.append(glove[s + win - 1])      # kinematics at window end
    return np.asarray(X, np.float32), np.asarray(Y, np.float32)

def build_arrays(recordings, subjects=None):
    """Window a set of recordings (optionally filtered to given subjects)."""
    Xs, Ys = [], []
    for emg, glove, subj in recordings:
        if subjects is not None and subj not in subjects:
            continue
        X, Y = window_recording(emg, glove, cfg.fs, cfg.win_ms, cfg.stride_ms)
        if len(X):
            Xs.append(X); Ys.append(Y)
    return np.concatenate(Xs), np.concatenate(Ys)

class Normalizer:
    """Per-channel z-score. Fit on train only, then apply everywhere."""
    def fit(self, X, Y):
        self.x_mean = X.mean((0, 2), keepdims=True)
        self.x_std = X.std((0, 2), keepdims=True) + 1e-8
        self.y_mean = Y.mean(0, keepdims=True)
        self.y_std = Y.std(0, keepdims=True) + 1e-8
        return self
    def x(self, X):     return (X - self.x_mean) / self.x_std
    def y(self, Y):     return (Y - self.y_mean) / self.y_std
    def y_inv(self, Yn): return Yn * self.y_std + self.y_mean

class EMGKinematicsDataset(Dataset):
    """(emg_window, kinematics_target) pairs for regression."""
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X); self.Y = torch.from_numpy(Y)
    def __len__(self):  return len(self.X)
    def __getitem__(self, i): return self.X[i], self.Y[i]

class EMGOnlyDataset(Dataset):
    """EMG windows only, for self-supervised pretraining (no labels)."""
    def __init__(self, X):
        self.X = torch.from_numpy(X)
    def __len__(self):  return len(self.X)
    def __getitem__(self, i): return self.X[i]

## 5. Models

A shared 1D-CNN encoder over raw EMG, with two heads:
* **MaskedAutoencoder** — SSL pretraining (reconstruct masked EMG, no labels).
* **RegressionModel** — supervised EMG &rarr; 22 kinematic values.

Pretrain the encoder with the autoencoder, then load those weights into the regression model. Training the regressor *without* loading them is the "from scratch" baseline.

In [5]:
class EMGEncoder(nn.Module):
    """(B, n_emg, T) -> feature map (B, feat_dim, T/4)."""
    def __init__(self, n_emg=12, feat_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_emg, 32, 5, padding=2), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, feat_dim, 3, padding=1), nn.BatchNorm1d(feat_dim), nn.ReLU(),
        )
        self.feat_dim = feat_dim
    def forward(self, x):
        return self.net(x)
    def pooled(self, x):
        return F.adaptive_avg_pool1d(self.net(x), 1).flatten(1)  # (B, feat_dim)

class _Decoder(nn.Module):
    """Feature map -> reconstructed EMG (B, n_emg, T)."""
    def __init__(self, feat_dim, n_emg):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(feat_dim, 64, 3, padding=1), nn.ReLU(),
            nn.Conv1d(64, 32, 3, padding=1), nn.ReLU(),
            nn.Conv1d(32, n_emg, 3, padding=1),
        )
    def forward(self, feat, out_len):
        y = self.net(feat)
        return F.interpolate(y, size=out_len, mode="linear", align_corners=False)

class MaskedAutoencoder(nn.Module):
    def __init__(self, n_emg=12, feat_dim=128):
        super().__init__()
        self.encoder = EMGEncoder(n_emg, feat_dim)
        self.decoder = _Decoder(feat_dim, n_emg)
    def forward(self, x_masked, out_len):
        return self.decoder(self.encoder(x_masked), out_len)

class RegressionModel(nn.Module):
    def __init__(self, n_emg=12, n_kin=22, feat_dim=128):
        super().__init__()
        self.encoder = EMGEncoder(n_emg, feat_dim)
        self.head = nn.Sequential(
            nn.Linear(feat_dim, feat_dim), nn.ReLU(),
            nn.Linear(feat_dim, n_kin),
        )
    def forward(self, x):
        return self.head(self.encoder.pooled(x))

## 6. Self-supervised pretraining (masked reconstruction)

Randomly zero out a fraction of each window's timesteps and train the encoder+decoder to reconstruct the masked parts. No labels — runs on all training-subject EMG. Saves the encoder for fine-tuning.

In [6]:
def random_mask(x, mask_ratio):
    """x: (B, C, T). Returns masked input and keep-mask (1=kept, 0=masked)."""
    B, C, T = x.shape
    keep = (torch.rand(B, 1, T, device=x.device) > mask_ratio).float()
    return x * keep, keep

def run_ssl_pretraining():
    recordings = load_files(cfg.data_dir)
    train_subjects = {s for *_, s in recordings if s not in cfg.test_subjects}
    X, _ = build_arrays(recordings, subjects=train_subjects)

    norm = Normalizer().fit(X, np.zeros((1, cfg.n_kin), np.float32))
    X = norm.x(X).astype(np.float32)

    loader = DataLoader(EMGOnlyDataset(X), batch_size=cfg.batch_size, shuffle=True, drop_last=True)
    model = MaskedAutoencoder(cfg.n_emg, cfg.feat_dim).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.ssl_lr)

    for ep in range(cfg.ssl_epochs):
        total = 0.0
        for xb in loader:
            xb = xb.to(device)
            x_masked, keep = random_mask(xb, cfg.mask_ratio)
            recon = model(x_masked, out_len=xb.shape[2])
            masked = 1.0 - keep
            loss = (F.mse_loss(recon, xb, reduction="none") * masked).sum() / (masked.sum() * xb.shape[1] + 1e-8)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item()
        print(f"[ssl] epoch {ep+1:3d}/{cfg.ssl_epochs}  loss {total/len(loader):.5f}")

    torch.save(model.encoder.state_dict(), "encoder_ssl.pt")
    print("saved -> encoder_ssl.pt")
    return model.encoder

In [7]:
# Run SSL pretraining (needs the data in ./data). Produces encoder_ssl.pt
run_ssl_pretraining()

loaded 12 recordings, subjects: [10, 11, 12, 13, 15, 16]
[ssl] epoch   1/40  loss 0.35041
[ssl] epoch   2/40  loss 0.28763
[ssl] epoch   3/40  loss 0.27128
[ssl] epoch   4/40  loss 0.25985
[ssl] epoch   5/40  loss 0.25483
[ssl] epoch   6/40  loss 0.24780
[ssl] epoch   7/40  loss 0.24536
[ssl] epoch   8/40  loss 0.24234
[ssl] epoch   9/40  loss 0.23736
[ssl] epoch  10/40  loss 0.23687
[ssl] epoch  11/40  loss 0.23319
[ssl] epoch  12/40  loss 0.23324
[ssl] epoch  13/40  loss 0.23146
[ssl] epoch  14/40  loss 0.22807
[ssl] epoch  15/40  loss 0.22855
[ssl] epoch  16/40  loss 0.22627
[ssl] epoch  17/40  loss 0.22457
[ssl] epoch  18/40  loss 0.22353
[ssl] epoch  19/40  loss 0.22248
[ssl] epoch  20/40  loss 0.22062
[ssl] epoch  21/40  loss 0.22168
[ssl] epoch  22/40  loss 0.21724
[ssl] epoch  23/40  loss 0.21871
[ssl] epoch  24/40  loss 0.21693
[ssl] epoch  25/40  loss 0.21664
[ssl] epoch  26/40  loss 0.21551
[ssl] epoch  27/40  loss 0.21452
[ssl] epoch  28/40  loss 0.21482
[ssl] epoch  29/40 

RuntimeError: File encoder_ssl.pt cannot be opened.

## 7. Supervised regression: EMG &rarr; kinematics

Two runs make the core experiment (objective 4):
* **from scratch** — `train_regressor(pretrained=None)`
* **SSL-initialised** — `train_regressor(pretrained="encoder_ssl.pt")`

The test set is entirely held-out subjects, so this measures **cross-subject generalisation**.

In [ ]:
def prepare_data():
    recordings = load_files(cfg.data_dir)
    all_subjects = {s for *_, s in recordings}
    train_subjects = all_subjects - set(cfg.test_subjects)

    Xtr, Ytr = build_arrays(recordings, subjects=train_subjects)
    Xte, Yte = build_arrays(recordings, subjects=set(cfg.test_subjects))

    norm = Normalizer().fit(Xtr, Ytr)   # fit on TRAIN only
    Xtr, Ytr = norm.x(Xtr).astype(np.float32), norm.y(Ytr).astype(np.float32)
    Xte, Yte = norm.x(Xte).astype(np.float32), norm.y(Yte).astype(np.float32)
    return (Xtr, Ytr), (Xte, Yte), norm

def train_regressor(pretrained=None, out="regressor.pt"):
    (Xtr, Ytr), (Xte, Yte), norm = prepare_data()
    tl = DataLoader(EMGKinematicsDataset(Xtr, Ytr), batch_size=cfg.batch_size, shuffle=True)
    te = DataLoader(EMGKinematicsDataset(Xte, Yte), batch_size=cfg.batch_size)

    model = RegressionModel(cfg.n_emg, cfg.n_kin, cfg.feat_dim).to(device)
    if pretrained:
        model.encoder.load_state_dict(torch.load(pretrained, map_location=device))
        print(f"loaded SSL encoder from {pretrained}")
    else:
        print("training from scratch (no SSL)")

    opt = torch.optim.Adam(model.parameters(), lr=cfg.reg_lr)
    lossf = nn.MSELoss()

    for ep in range(cfg.reg_epochs):
        model.train()
        for xb, yb in tl:
            xb, yb = xb.to(device), yb.to(device)
            loss = lossf(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval(); preds, gts = [], []
        with torch.no_grad():
            for xb, yb in te:
                preds.append(model(xb.to(device)).cpu().numpy()); gts.append(yb.numpy())
        preds, gts = np.concatenate(preds), np.concatenate(gts)
        print(f"[reg] epoch {ep+1:3d}/{cfg.reg_epochs}  RMSE {rmse(gts,preds):.4f}  R2 {r2(gts,preds):.3f}  corr {mean_corr(gts,preds):.3f}")

    torch.save({"state_dict": model.state_dict(),
                "y_mean": norm.y_mean, "y_std": norm.y_std,
                "x_mean": norm.x_mean, "x_std": norm.x_std}, out)
    print(f"saved -> {out}")
    return model

In [ ]:
# Baseline: from scratch
train_regressor(pretrained=None, out="scratch.pt")

In [ ]:
# SSL-initialised (run the pretraining cell first so encoder_ssl.pt exists)
train_regressor(pretrained="encoder_ssl.pt", out="ssl.pt")

## 8. Evaluation (objective 4)

Metrics on the held-out subjects, plus a noise-robustness sweep. Run it on **both** models and put the two tables side by side in your report — the gap is your main result.

In [ ]:
def load_model(path):
    ckpt = torch.load(path, map_location=device)
    model = RegressionModel(cfg.n_emg, cfg.n_kin, cfg.feat_dim).to(device)
    model.load_state_dict(ckpt["state_dict"]); model.eval()
    stats = {k: ckpt[k] for k in ("x_mean", "x_std", "y_mean", "y_std")}
    return model, stats

def predict(model, X):
    out = []
    with torch.no_grad():
        for i in range(0, len(X), cfg.batch_size):
            xb = torch.from_numpy(X[i:i+cfg.batch_size]).to(device)
            out.append(model(xb).cpu().numpy())
    return np.concatenate(out)

def evaluate_model(path):
    model, stats = load_model(path)
    recordings = load_files(cfg.data_dir)
    Xte_raw, Yte = build_arrays(recordings, subjects=set(cfg.test_subjects))
    Yte_n = (Yte - stats["y_mean"]) / stats["y_std"]

    Xte = ((Xte_raw - stats["x_mean"]) / stats["x_std"]).astype(np.float32)
    pred = predict(model, Xte)
    print(f"=== {path}: clean (held-out subjects) ===")
    print(f"RMSE {rmse(Yte_n,pred):.4f}   R2 {r2(Yte_n,pred):.3f}   corr {mean_corr(Yte_n,pred):.3f}")

    print("\n=== SNR robustness ===")
    print(f"{'SNR(dB)':>8} {'RMSE':>8} {'R2':>8} {'corr':>8}")
    for snr in (20, 15, 10, 5, 0):
        Xn = add_noise_snr(Xte_raw, snr)
        Xn = ((Xn - stats["x_mean"]) / stats["x_std"]).astype(np.float32)
        p = predict(model, Xn)
        print(f"{snr:>8} {rmse(Yte_n,p):>8.4f} {r2(Yte_n,p):>8.3f} {mean_corr(Yte_n,p):>8.3f}")

In [ ]:
evaluate_model("scratch.pt")
print("\n" + "="*50 + "\n")
evaluate_model("ssl.pt")

## 9. Visualization
Predicted vs. true joint trajectories on a held-out subject — the one supporting figure the brief needs.

In [ ]:
def plot_predictions(path="ssl.pt", joints=(0,1,2,3), n=500):
    model, stats = load_model(path)
    recordings = load_files(cfg.data_dir)
    Xraw, Y = build_arrays(recordings, subjects=set(cfg.test_subjects))
    Xn = ((Xraw - stats["x_mean"]) / stats["x_std"]).astype(np.float32)
    pred = predict(model, Xn[:n]) * stats["y_std"] + stats["y_mean"]
    true = Y[:n]

    fig, axes = plt.subplots(len(joints), 1, figsize=(9, 2*len(joints)), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, j in zip(axes, joints):
        ax.plot(true[:, j], label="true", lw=1.2)
        ax.plot(pred[:, j], label="predicted", lw=1.2, alpha=0.8)
        ax.set_ylabel(f"joint {j}")
    axes[0].legend(loc="upper right")
    axes[-1].set_xlabel("window index (time)")
    fig.suptitle("Predicted vs. true hand kinematics (held-out subject)")
    fig.tight_layout()
    fig.savefig("kinematics_prediction.png", dpi=130)
    plt.show()
    print("saved -> kinematics_prediction.png")

plot_predictions("ssl.pt")

## 10. Notes for the write-up

* **RMSE** = average joint-angle error (lower better); **R²** = variance explained; **corr** = mean Pearson correlation of predicted vs true joint tracks.
* The **from-scratch vs SSL** comparison on **held-out subjects** is your central result — does SSL improve cross-subject prediction?
* The **SNR sweep** is your robustness result and the place SSL is expected to help most.
* Known limitations worth discussing: cross-subject drop-off, weak cases (e.g. all-fingers-closed), and any channel-scaling quirks in the glove data.
* This notebook is the graded algorithm + evaluation. A 3D hand posed from these predictions is an optional demo, not part of the graded core.